# First API Call: Streaming, Tokens, and the Response Object

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/03-first-api-call.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You wire up your first AI feature. It works fine in your test notebook: you call the API, get a response, display it. You ship it. Users complain the page freezes for 4 seconds, then text appears all at once. Someone on your team suggests "just add a loading spinner." You add the spinner. Users still complain: they can see the response starting to form elsewhere and want to read along as it generates.

The fix is streaming. But streaming is not a drop-in swap -- you have to accumulate the chunks, handle the stop event, and count tokens differently. The second problem is that even on non-streaming calls, most engineers look only at the text and ignore the metadata: token counts, stop reasons,...

## The Concept

### The Request-Response Cycle

```
NON-STREAMING (blocks until complete):

Client                          Anthropic API
  |                                   |
  |--- POST /v1/messages ------------>|
  |                                   | (model generates entire response)
  |                                   | (might take 3-8 seconds for long outputs)
  |<-- 200 OK + full response --------|
  |                                   |
  Display text                        |

STREAMING (tokens arrive as generated):

Client                          Anthropic API
  |                               ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 03: First API Call - Streaming, Tokens, and the Response Object

Demonstrates:
- Non-streaming call with full response object inspection
- Stop reason detection (end_turn vs max_tokens)
- Streaming call with chunk accumulation
- Pre-flight token counting
- Production streaming helper pattern

Run with: uv run python main.py
"""

import os
import anthropic
from dotenv import load_dotenv


def inspect_response(response: anthropic.types.Message) -> None:
    """Print all fields of a Message response object."""
    print("=== Response Object Fields ===")
    print(f"  id:            {response.id}")
    print(f"  type:          {response.type}")
    print(f"  role:          {response.role}")
    print(f"  model:         {response.model}")
    print(f"  stop_reason:   {response.stop_reason}")
    print(f"  stop_sequence: {response.stop_sequence}")
    print(f"  input_tokens:  {response.usage.input_tokens}")
    print(f"  output_tokens: {response.usage.output_tokens}")
    print(f"  content blocks:{len(response.content)}")
    for i, block in enumerate(response.content):
        print(f"    [{i}] type={block.type}, text_length={len(block.text) if block.type == 'text' else 'N/A'}")
    print(f"\n  Text:\n  {response.content[0].text[:200]}")


def safe_extract_text(response: anthropic.types.Message) -> str:
    """
    Extract text from response, raising if truncated (stop_reason == max_tokens).
    In production, truncation is a bug -- never silently return partial output.
    """
    if response.stop_reason == "max_tokens":
        raise RuntimeError(
            f"Response truncated at {response.usage.output_tokens} output tokens. "
            "Increase max_tokens or split the task into smaller pieces."
        )
    if not response.content or response.content[0].type != "text":
        raise ValueError(f"Unexpected response content: {response.content}")
    return response.content[0].text


def stream_to_stdout(client: anthropic.Anthropic, prompt: str) -> tuple[str, anthropic.types.Usage]:
    """
    Stream a response to stdout, return (full_text, usage).
    This is the production pattern for server-sent events in a web API.
    """
    accumulated: list[str] = []

    print("=== Streaming (tokens appear as generated) ===")
    with client.messages.stream(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for text_chunk in stream.text_stream:
            print(text_chunk, end="", flush=True)
            accumulated.append(text_chunk)

    print()  # newline after stream
    final_message = stream.get_final_message()
    return "".join(accumulated), final_message.usage


def demonstrate_max_tokens_truncation(client: anthropic.Anthropic) -> None:
    """Show what max_tokens truncation looks like in the response object."""
    print("\n=== max_tokens Truncation Demo ===")
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=10,  # intentionally tiny to force truncation
        messages=[
            {"role": "user", "content": "Write a 100-word explanation of how APIs work."}
        ],
    )
    print(f"stop_reason: {response.stop_reason}")  # "max_tokens"
    print(f"output_tokens generated: {response.usage.output_tokens}")
    print(f"truncated text: '{response.content[0].text}'")
    print("NOTE: this response is incomplete -- increase max_tokens in production")


def count_tokens_preflight(client: anthropic.Anthropic, prompt: str) -> int:
    """Count tokens before sending to catch context window violations."""
    count = client.messages.count_tokens(
        model="claude-3-5-haiku-20241022",
        messages=[{"role": "user", "content": prompt}],
    )
    return count.input_tokens


def main() -> None:
    load_dotenv()

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("ERROR: ANTHROPIC_API_KEY not set. Complete Lesson 02 first.")
        return

    client = anthropic.Anthropic(api_key=api_key)
    prompt = "Explain what a context window is in one sentence."

    # 1. Pre-flight token count
    print("=== Pre-flight Token Count ===")
    token_count = count_tokens_preflight(client, prompt)
    print(f"Input tokens for this prompt: {token_count}")
    print(f"Context window remaining: {200_000 - token_count:,} tokens")

    # 2. Non-streaming call with full inspection
    print("\n=== Non-Streaming Call ===")
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    inspect_response(response)

    # 3. Safe text extraction
    print("\n=== Safe Text Extraction ===")
    text = safe_extract_text(response)
    print(f"Extracted text: {text[:100]}...")

    # 4. Streaming the same prompt
    streamed_text, usage = stream_to_stdout(client, prompt)
    print(f"\nStream stats:")
    print(f"  input_tokens:  {usage.input_tokens}")
    print(f"  output_tokens: {usage.output_tokens}")
    print(f"  accumulated characters: {len(streamed_text)}")

    # 5. Truncation demo
    demonstrate_max_tokens_truncation(client)

    # 6. Compare token counts: non-streaming vs streaming
    print("\n=== Token Count Comparison ===")
    print(f"Non-streaming: {response.usage.input_tokens} in / {response.usage.output_tokens} out")
    print(f"Streaming:     {usage.input_tokens} in / {usage.output_tokens} out")
    print("(Should be identical for the same prompt)")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which field in the response object tells you definitively that truncation occurred, and what value should it have?**

- A. response.content[0].text ends with '...' to signal truncation
- B. response.stop_reason equals 'max_tokens'
- C. response.usage.output_tokens equals 128
- D. response.id contains 'truncated' in the string

**2. What is the correct change to make in your API call code?**

- A. Set a shorter timeout parameter on the API call to force faster responses
- B. Switch from client.messages.create() to client.messages.stream() and yield chunks as they arrive
- C. Use a smaller model (Haiku instead of Sonnet) to get faster responses
- D. Reduce max_tokens to force shorter responses that arrive sooner

**3. **

- A. Read usage from each individual chunk in the stream loop
- B. Call stream.get_final_message() after the `with` block exits to get usage from the final message
- C. Usage is not available in streaming mode -- use non-streaming for cost accounting
- D. Call client.messages.count_tokens() before and after the stream

**4. Which combination of fields covers all three requirements?**

- A. response.model, response.stop_reason, response.usage
- B. response.id, response.content[0].text, response.usage
- C. response.model, response.id, response.usage.input_tokens
- D. response.type, response.role, response.usage.output_tokens

**5. What is the correct way to count tokens without making a full API call?**

- A. Estimate by dividing character count by 4 (approximately 4 chars per token)
- B. Make a test API call with max_tokens=1 and read usage.input_tokens from the response
- C. Use client.messages.count_tokens() with the model and messages parameters
- D. Use the tiktoken library which works for all models including Claude

**6. What is the correct architectural fix?**

- A. Increase the server timeout to 120 seconds
- B. Reduce max_tokens to limit response length to under 500 tokens
- C. Switch to streaming and return a Server-Sent Events (SSE) or chunked response from the endpoint
- D. Move the API call to a background worker and poll for results

**7. What is the most likely explanation?**

- A. Token counting is non-deterministic and varies randomly
- B. The model version was updated between calls and uses a slightly different tokenizer
- C. Your system prompt or conversation history grew between the two calls
- D. Input token counts include a random sampling component

_Answers are in `checks.json` in the lesson directory._